# Project 3: Disaster Mapping using SAR - Odisha Cyclone Zone
## Notebook 01: Google Earth Engine Data Extraction & Patch Generation

**Objective:** In this notebook, we will authenticate Google Earth Engine (GEE), define our Region of Interest (ROI) over the coastal Odisha cyclone zone, filter Sentinel-1 SAR images for pre-event (April 2019) and post-event (May 2019 - Cyclone Fani), apply a baseline threshold to create a target flood mask, and extract $256 \times 256$ patches as a `.zip` file.

In [3]:
import ee
import os

# Google Cloud Project ID manually pass karenge jo verification bypass karega
# Agar aapka koi explicit project ID hai toh yahan likho, nahi toh ye default search karega
try:
    print("Attempting project-based initialization...")
    ee.Initialize(project='iron-airfoil-496807-j7') # Apna GEE project name daal sakte ho agar pata ho
    print("🔥 BOOM! Initialization Successful via Project ID!")
except Exception:
    print("Project method failed. Launching secondary authenticater...")
    from google.colab import auth
    auth.authenticate_user() # Ye Colab ke standard Google account ko bind kar dega

    import google.auth
    credentials, project = google.auth.default()
    ee.Initialize(credentials=credentials, project=project)
    print("🔥 SUCCESS! Google Earth Engine successfully initialized using Colab Identity!")

Attempting project-based initialization...
🔥 BOOM! Initialization Successful via Project ID!


## Step 1: Define Region of Interest (ROI) & Filter Sentinel-1 Images

We will define a rectangular bounding box covering coastal Odisha (Puri and Jagatsinghpur districts). We will then load the Sentinel-1 Ground Range Detected (GRD) dataset in Interferometric Wide (IW) mode with **VV Polarization**, which provides the best contrast for land-water separation.

In [12]:
# Updated Bounding Box: Covering Puri, Jagatsinghpur, Kendrapara, Bhadrak & Balasore
odisha_roi = ee.Geometry.Rectangle([85.0, 19.5, 87.2, 21.8])

# Load Sentinel-1 GRD Collection
s1_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filterBounds(odisha_roi)

# 1. Pre-Event Image Baseline (April 2019 - Using .mean() for stability)
pre_scene = s1_collection.filterDate('2019-04-10', '2019-04-25').mean().clip(odisha_roi).select('VV')

# 2. Post-Event Image (May 2019 - Cyclone Fani Landfall & Flooding)
post_scene = s1_collection.filterDate('2019-05-04', '2019-05-10').mean().clip(odisha_roi).select('VV')

print("🔥 Perfect! Bhadrak, Balasore, Jagatsinghpur, and Puri regions loaded as pre_scene and post_scene!")

🔥 Perfect! Bhadrak, Balasore, Jagatsinghpur, and Puri regions loaded as pre_scene and post_scene!


## Step 2: Creating the Target Flood Mask (Ground Truth Baseline)

Smooth open water surfaces cause specular reflection, sending very low radar signals back to the satellite. This results in dark pixels (low backscatter values). Based on NASA ARSET guidelines, a standard backscatter threshold of **-18 dB** or lower perfectly identifies flooded regions. We will use this to generate our binary target masks ($1 = \text{Flood}, 0 = \text{No-Flood}$).

In [13]:
# Create Binary Flood Mask using a threshold approach
# Pixels below -18 dB in the post-event image that were NOT water before are flagged as flood
# CRITICAL FIX: Cast to .float() to prevent GEE Export 'inconsistent types' error code 3
flood_mask = post_scene.lt(-18).float().rename('flood_label')

print("🔥 Success! Binary target flood mask generated and cast to Float successfully!")

🔥 Success! Binary target flood mask generated and cast to Float successfully!


## Step 3: Generating $256 \times 256$ Patches and Exporting as ZIP

Since deep learning models like U-Net cannot ingest massive satellite scenes directly due to memory constraints, we will break down our ROI into smaller, manageable $256 \times 256$ pixel patches. We will package these patches into a single compressed `.zip` archive ready to be uploaded to Kaggle.

In [15]:
# CRITICAL STRIP: Force ALL bands to be strictly Float32 (.float()) to prevent inconsistency
pre_fixed = pre_scene.float()
post_fixed = post_scene.float()
mask_fixed = flood_mask.float()

# Ab stack karenge bina kisi tension ke!
final_stack = ee.Image.cat([
    pre_fixed.rename('pre_vv'),
    post_fixed.rename('post_vv'),
    mask_fixed.rename('flood_label')
])

print("Stacking complete with synchronized Float32 types. Starting background export...")

# Exporting the master multi-band stack to Google Drive
optimized_task = ee.batch.Export.image.toDrive(
    image=final_stack,
    description='Odisha_SAR_Cyclone_Belt_Fixed',
    folder='EarthEngine_Outputs',
    fileNamePrefix='odisha_stacked_master_fixed',
    scale=40,
    region=odisha_roi,
    fileFormat='GeoTIFF',
    maxPixels=1e13
)

# Start the background cloud operation
optimized_task.start()
print("🚀 Task submitted to Google Cloud servers!")

import time
# Status Monitor Loop
while optimized_task.active():
    print(f"Waiting... Current State: {optimized_task.status()['state']} (Server processing...)")
    time.sleep(20)

print(f"\n🔥 FINAL STATUS: {optimized_task.status()['state']}")
if optimized_task.status()['state'] == 'COMPLETED':
    print("BOOM! Ab bina kisi error ke Google Drive check karo bhai, file mil jayegi!")

Stacking complete with synchronized Float32 types. Starting background export...
🚀 Task submitted to Google Cloud servers!
Waiting... Current State: READY (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State: RUNNING (Server processing...)
Waiting... Current State